In [11]:
from pathlib import Path
from datetime import datetime
import sys
import time
import json
import pandas as pd

PROJECT_ROOT = Path("/home/harielpadillasanchez/Documentos/TT/TT2")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from configs.models import MODELS
from src.experiment.runner import ExperimentRunner
from src.evaluation.metrics import evaluate_dataframe, summarize_metrics

In [12]:
DATA_DIR = PROJECT_ROOT / "data"
SPLIT_DIR = DATA_DIR / "splits"
OUT_DIR = PROJECT_ROOT / "outputs" / "prompt_final_eval"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TEST_PATH = SPLIT_DIR / "feina_clean_30_test.csv"

print("TEST_PATH:", TEST_PATH)
print("OUT_DIR:", OUT_DIR)

TEST_PATH: /home/harielpadillasanchez/Documentos/TT/TT2/data/splits/feina_clean_30_test.csv
OUT_DIR: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/prompt_final_eval


In [13]:
df_test = pd.read_csv(TEST_PATH)

print("Shape df_test:", df_test.shape)
display(df_test.head(3))
display(df_test.columns.tolist())

Shape df_test: (238, 15)


,row_id,idFinal,source_text,reference_text,idcod,atr0,atr1,atr2,atr3,atr4,atr5,atr6,atr7,atr8,lex
0,75,5329b_LibroBAC.pdf,Todo lo anterior está ligado también al desord...,Lo anterior se relaciona con que muchas person...,Fiorella,15.0,19.0,3.0,5.0,16.0,NaN,NaN,NaN,NaN,0
1,93,7643_LibroBAC.pdf,Este comportamiento puede obedecer a varias ca...,"Este comportamiento tiene varias causas, aunqu...",Fiorella,15.0,5.0,1.0,4.0,16.0,17.0,19.0,2.0,NaN,1
2,94,7649_LibroBAC.pdf,"Dichas personas, ante sus temores, prefieren v...",Dichas personas prefieren vivir en la oscurida...,Fiorella,21.0,15.0,2.0,4.0,17.0,19.0,5.0,NaN,NaN,0


['row_id',
 'idFinal',
 'source_text',
 'reference_text',
 'idcod',
 'atr0',
 'atr1',
 'atr2',
 'atr3',
 'atr4',
 'atr5',
 'atr6',
 'atr7',
 'atr8',
 'lex']

In [14]:
PROMPT_WINNER_CONFIGS = [
    {
        "model_key": "llama3",
        "config_label": "llama3_cfg_3",
        "ruleset": "R1",
        "temperature": 0.3,
        "top_p": 0.90,
        "repetition_penalty": 1.15,
        "max_new_tokens": 256,
    }
]

display(pd.DataFrame(PROMPT_WINNER_CONFIGS))

,model_key,config_label,ruleset,temperature,top_p,repetition_penalty,max_new_tokens
0,llama3,llama3_cfg_3,R1,0.3,0.9,1.15,256


In [15]:
PROMPT_TYPE = "zero-shot"
FEW_SHOT_EXAMPLES = None
FEW_SHOT_EXAMPLE_IDS = []

In [20]:
def run_prompt_eval_on_split(
    df_split: pd.DataFrame,
    split_name: str,
    candidate_configs: list,
    prompt_type: str,
    few_shot_examples: list | None,
    few_shot_example_ids: list | None,
    output_dir: Path,
):
    run_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    experiment_id = f"exp_prompt_final_{split_name}_{run_ts}"

    runner = ExperimentRunner(
        experiment_id=experiment_id,
        log_dir=str(PROJECT_ROOT / "outputs" / "logs")
    )

    records = []
    total_runs = len(df_split) * len(candidate_configs)
    run_counter = 0

    t0 = time.perf_counter()

    for cfg in candidate_configs:
        for _, row in df_split.iterrows():
            run_counter += 1
            print(
                f"[{run_counter}/{total_runs}] "
                f"split={split_name} | model={cfg['model_key']} | "
                f"cfg={cfg['config_label']} | ruleset={cfg['ruleset']} | "
                f"sample_id={row['row_id']}"
            )

            record = runner.run_one(
                dataset_name=f"feina_clean_30_{split_name}",
                model_key=cfg["model_key"],
                prompt_type=prompt_type,
                ruleset=cfg["ruleset"],
                source_text=row["source_text"],
                reference_text=row["reference_text"],
                sample_id=str(row["row_id"]),
                fold_id=None,
                split_name=split_name,
                few_shot_examples=few_shot_examples if prompt_type == "few-shot" else None,
                few_shot_example_ids=few_shot_example_ids if prompt_type == "few-shot" else None,
                generation_config={
                    "temperature": cfg["temperature"],
                    "top_p": cfg["top_p"],
                    "repetition_penalty": cfg["repetition_penalty"],
                    "max_new_tokens": cfg["max_new_tokens"],
                },
            )

            record_dict = record.to_dict()
            record_dict["config_label"] = cfg["config_label"]

            for meta_col in ["idFinal", "idcod", "lex", "atr0", "atr1", "atr2", "atr3", "atr4", "atr5", "atr6", "atr7", "atr8"]:
                if meta_col in row.index:
                    record_dict[meta_col] = row[meta_col]

            records.append(record_dict)

    t1 = time.perf_counter()
    print(f"Tiempo total inferencia split={split_name}: {(t1 - t0)/60:.2f} min")

    raw_df = pd.DataFrame(records)

    if "status" in raw_df.columns:
        eval_input_df = raw_df[raw_df["status"] == "success"].copy()
    else:
        eval_input_df = raw_df.copy()

    evaluated_df = evaluate_dataframe(
        eval_input_df,
        source_col="source_text",
        pred_col="generated_text",
        ref_col="reference_text",
    )

    summary_df = summarize_metrics(
        evaluated_df,
        group_cols=[],
    )

    cfg = candidate_configs[0]
    summary_df["model_key"] = cfg["model_key"]
    summary_df["config_label"] = cfg["config_label"]
    summary_df["ruleset"] = cfg["ruleset"]
    summary_df["temperature"] = cfg["temperature"]
    summary_df["top_p"] = cfg["top_p"]
    summary_df["repetition_penalty"] = cfg["repetition_penalty"]
    summary_df["max_new_tokens"] = cfg["max_new_tokens"]
    summary_df["n_rows_evaluated"] = len(evaluated_df)

    display(summary_df)

    raw_path = output_dir / f"{experiment_id}_raw_results.csv"
    eval_path = output_dir / f"{experiment_id}_evaluated.csv"
    summary_path = output_dir / f"{experiment_id}_summary.csv"
    meta_path = output_dir / f"{experiment_id}_metadata.json"

    raw_df.to_csv(raw_path, index=False, encoding="utf-8-sig")
    evaluated_df.to_csv(eval_path, index=False, encoding="utf-8-sig")
    summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

    metadata = {
        "experiment_id": experiment_id,
        "split_name": split_name,
        "n_rows_split": len(df_split),
        "n_candidate_configs": len(candidate_configs),
        "expected_total_runs": total_runs,
        "runtime_seconds_total": t1 - t0,
        "raw_path": str(raw_path),
        "eval_path": str(eval_path),
        "summary_path": str(summary_path),
    }

    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    return {
        "experiment_id": experiment_id,
        "raw_df": raw_df,
        "evaluated_df": evaluated_df,
        "summary_df": summary_df,
        "metadata": metadata,
    }

In [21]:
t0 = time.perf_counter()

prompt_test_results = run_prompt_eval_on_split(
    df_split=df_test,
    split_name="test30_prompt_winner",
    candidate_configs=PROMPT_WINNER_CONFIGS,
    prompt_type=PROMPT_TYPE,
    few_shot_examples=FEW_SHOT_EXAMPLES,
    few_shot_example_ids=FEW_SHOT_EXAMPLE_IDS,
    output_dir=OUT_DIR,
)

t1 = time.perf_counter()
print(f"Tiempo total evaluación prompt-based test: {(t1 - t0)/60:.2f} min")

[1/238] split=test30_prompt_winner | model=llama3 | cfg=llama3_cfg_3 | ruleset=R1 | sample_id=75
[2/238] split=test30_prompt_winner | model=llama3 | cfg=llama3_cfg_3 | ruleset=R1 | sample_id=93
[3/238] split=test30_prompt_winner | model=llama3 | cfg=llama3_cfg_3 | ruleset=R1 | sample_id=94
[4/238] split=test30_prompt_winner | model=llama3 | cfg=llama3_cfg_3 | ruleset=R1 | sample_id=96
[5/238] split=test30_prompt_winner | model=llama3 | cfg=llama3_cfg_3 | ruleset=R1 | sample_id=123
[6/238] split=test30_prompt_winner | model=llama3 | cfg=llama3_cfg_3 | ruleset=R1 | sample_id=136
[7/238] split=test30_prompt_winner | model=llama3 | cfg=llama3_cfg_3 | ruleset=R1 | sample_id=150
[8/238] split=test30_prompt_winner | model=llama3 | cfg=llama3_cfg_3 | ruleset=R1 | sample_id=169
[9/238] split=test30_prompt_winner | model=llama3 | cfg=llama3_cfg_3 | ruleset=R1 | sample_id=185
[10/238] split=test30_prompt_winner | model=llama3 | cfg=llama3_cfg_3 | ruleset=R1 | sample_id=200
[11/238] split=test30_p

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 1c4a7f79-dddd-4e2a-b9d9-af6b3e3593c4)')' thrown while requesting HEAD https://huggingface.co/bert-base-multilingual-cased/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


,sari,bleu,fernandez_huerta_pred,fernandez_huerta_src,fernandez_huerta_delta,compression_ratio_eval,sentence_splits,levenshtein_similarity,exact_copy,additions_proportion,...,inflesz_delta,bertscore_f1,model_key,config_label,ruleset,temperature,top_p,repetition_penalty,max_new_tokens,n_rows_evaluated
0,30.320823,9.123028,77.364982,76.029628,1.335355,0.819666,0.344538,0.428487,0.0,0.591147,...,16.457085,0.78851,llama3,llama3_cfg_3,R1,0.3,0.9,1.15,256,238


Tiempo total evaluación prompt-based test: 31.50 min


In [22]:
prompt_test_summary = prompt_test_results["summary_df"].copy()
display(prompt_test_summary)
print(prompt_test_summary.columns.tolist())

,sari,bleu,fernandez_huerta_pred,fernandez_huerta_src,fernandez_huerta_delta,compression_ratio_eval,sentence_splits,levenshtein_similarity,exact_copy,additions_proportion,...,inflesz_delta,bertscore_f1,model_key,config_label,ruleset,temperature,top_p,repetition_penalty,max_new_tokens,n_rows_evaluated
0,30.320823,9.123028,77.364982,76.029628,1.335355,0.819666,0.344538,0.428487,0.0,0.591147,...,16.457085,0.78851,llama3,llama3_cfg_3,R1,0.3,0.9,1.15,256,238


['sari', 'bleu', 'fernandez_huerta_pred', 'fernandez_huerta_src', 'fernandez_huerta_delta', 'compression_ratio_eval', 'sentence_splits', 'levenshtein_similarity', 'exact_copy', 'additions_proportion', 'deletions_proportion', 'rouge1_f', 'rouge2_f', 'rougeL_f', 'inflesz_pred', 'inflesz_src', 'inflesz_delta', 'bertscore_f1', 'model_key', 'config_label', 'ruleset', 'temperature', 'top_p', 'repetition_penalty', 'max_new_tokens', 'n_rows_evaluated']


In [23]:
if "leader_score" not in prompt_test_summary.columns and {"sari", "bertscore_f1"}.issubset(prompt_test_summary.columns):
    prompt_test_summary["leader_score"] = (
        0.6 * prompt_test_summary["sari"] +
        0.4 * (prompt_test_summary["bertscore_f1"] * 100.0)
    )

display(prompt_test_summary)

,sari,bleu,fernandez_huerta_pred,fernandez_huerta_src,fernandez_huerta_delta,compression_ratio_eval,sentence_splits,levenshtein_similarity,exact_copy,additions_proportion,...,bertscore_f1,model_key,config_label,ruleset,temperature,top_p,repetition_penalty,max_new_tokens,n_rows_evaluated,leader_score
0,30.320823,9.123028,77.364982,76.029628,1.335355,0.819666,0.344538,0.428487,0.0,0.591147,...,0.78851,llama3,llama3_cfg_3,R1,0.3,0.9,1.15,256,238,49.732899


In [24]:
prompt_summary_path = OUT_DIR / "prompt_winner_test30_summary.csv"
prompt_test_summary.to_csv(prompt_summary_path, index=False, encoding="utf-8-sig")

print("Resumen prompt-based guardado en:", prompt_summary_path)

Resumen prompt-based guardado en: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/prompt_final_eval/prompt_winner_test30_summary.csv
